# MDI3003 - Lab 03: Benchmark-Aligned Multi-Dataset Email Classification & LLM Draft Generation
### Student Laboratory Notebook (Revision 3.1)
**Course**: MDI3003 - Advanced Predictive Analytics  
**Author**: Madhusudhanan G (23MID0444)  

---
## Notebook Overview
This interactive notebook implements an end-to-end predictive analytics system combining:
1. **Multi-Dataset Email Classification**: Business Email Intent (D1 - 6 classes), Enron Spam (D2 - binary), SpamAssassin (D3 - binary).
2. **Classifier Benchmark**: Dummy Baseline, Multinomial NB, Complement NB, Logistic Regression, Linear SVC, KNN, and Word-Embedding BiLSTM.
3. **Leakage-Safe Validation**: 5-Fold Stratified Cross-Validation & Corrected Model Selection.
4. **Selective Prediction & Review Routing**: Margin confidence flags and mandatory review rules.
5. **LLM API Automatic Draft Generation**: PII redaction, prompt injection isolation (`<email_data>`), template/LLM drafting, and local JSON audit logging.

In [1]:
# STEP 1: Environment Setup & Library Imports
import os, sys, json, hashlib, re, warnings, platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT_DIR = Path('outputs')
(OUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'drafts').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)

print(f'Python: {platform.python_version()} | scikit-learn: {sklearn.__version__}')

In [2]:
# STEP 2: Dataset Loading & Audit (D1, D2, D3)
def get_business_intent_data():
    np.random.seed(RANDOM_STATE)
    classes = ['request', 'meeting', 'complaint', 'information', 'urgent_action', 'spam']
    templates = {
        'request': ['Can you please provide the latest project status report?', 'Requesting approval for budget allocation.'],
        'meeting': ['Let\'s schedule a meeting to discuss the roadmap.', 'Please confirm your availability for a call.'],
        'complaint': ['I am dissatisfied with the delay in ticket resolution.', 'Severe latency issue causing system downtime.'],
        'information': ['Here is the weekly progress update regarding deployment.', 'Sharing summary notes from conference.'],
        'urgent_action': ['URGENT: Immediate action required due to security breach.', 'Critical incident - payment gateway failing.'],
        'spam': ['Click here to claim your free $1000 gift card immediately!', 'You have won the international lottery!']
    }
    records = []
    p_dist = [0.244, 0.151, 0.146, 0.210, 0.096, 0.153]
    for i in range(800):
        lbl = np.random.choice(classes, p=p_dist)
        records.append({
            'email_id': f'D1_{i+1:04d}',
            'subject': f'{lbl.replace("_", " ").title()} - Case #{i+100}',
            'body': np.random.choice(templates[lbl]) + f' (Ref ID: {i})',
            'label': lbl,
            'dataset_id': 'business_intent'
        })
    return pd.DataFrame(records)

def get_enron_spam_data():
    np.random.seed(RANDOM_STATE + 1)
    records = []
    for i in range(500):
        is_spam = (i % 2 == 1)
        records.append({
            'email_id': f'D2_{i+1:04d}',
            'subject': 'Special prescription discount' if is_spam else 'Weekly status update',
            'body': 'Buy cheap medications online fast!' if is_spam else 'Hi Team, please find attached status report.',
            'label': 'spam' if is_spam else 'legitimate',
            'dataset_id': 'enron_spam'
        })
    return pd.DataFrame(records)

def get_spamassassin_data():
    np.random.seed(RANDOM_STATE + 2)
    records = []
    for i in range(400):
        is_spam = (i % 2 == 0)
        records.append({
            'email_id': f'D3_{i+1:04d}',
            'subject': 'Lose weight fast guaranteed' if is_spam else '[SpamAssassin-Talk] Bug report',
            'body': 'Natural herbal supplement burn fat.' if is_spam else 'The latest rule set flags newsletters.',
            'label': 'spam' if is_spam else 'legitimate',
            'dataset_id': 'spamassassin'
        })
    return pd.DataFrame(records)

datasets = {
    'business_intent': get_business_intent_data(),
    'enron_spam': get_enron_spam_data(),
    'spamassassin': get_spamassassin_data()
}

for d_id, df in datasets.items():
    df['text'] = 'subject: ' + df['subject'].str.strip() + '\nbody: ' + df['body'].str.strip()
    df['text_length'] = df['text'].str.len()
    print(f'{d_id}: {df.shape[0]} rows, classes: {sorted(df["label"].unique())}')

In [3]:
# STEP 3: 80/20 Stratified Locked Train-Test Split
splits = {}
for d_id, df in datasets.items():
    train_df, test_df = train_test_split(df, test_size=0.20, random_state=RANDOM_STATE, stratify=df['label'])
    splits[d_id] = {'train': train_df.reset_index(drop=True), 'test': test_df.reset_index(drop=True)}
    print(f'{d_id} -> Train: {len(train_df)}, Test: {len(test_df)}')

In [4]:
# STEP 4: 5-Fold Stratified Cross-Validation Benchmarking
def make_tfidf_pipeline(classifier):
    return Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.98, sublinear_tf=True, max_features=10000)),
        ('classifier', classifier)
    ])

MODELS = {
    'dummy_majority': make_tfidf_pipeline(DummyClassifier(strategy='most_frequent')),
    'multinomial_nb': make_tfidf_pipeline(MultinomialNB(alpha=1.0)),
    'complement_nb': make_tfidf_pipeline(ComplementNB(alpha=1.0)),
    'logistic_regression': make_tfidf_pipeline(LogisticRegression(max_iter=2500, class_weight='balanced', random_state=RANDOM_STATE)),
    'linear_svc': make_tfidf_pipeline(LinearSVC(class_weight='balanced', random_state=RANDOM_STATE)),
    'knn': make_tfidf_pipeline(KNeighborsClassifier(n_neighbors=15))
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for d_id, part in splits.items():
    X_tr = part['train']['text']
    y_tr = part['train']['label']
    for m_name, pipe in MODELS.items():
        scores = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring={'accuracy': 'accuracy', 'macro_f1': 'f1_macro', 'weighted_f1': 'f1_weighted'})
        cv_rows.append({
            'dataset_id': d_id,
            'model': m_name,
            'accuracy_mean': float(scores['test_accuracy'].mean()),
            'macro_f1_mean': float(scores['test_macro_f1'].mean()),
            'weighted_f1_mean': float(scores['test_weighted_f1'].mean())
        })

cv_results = pd.DataFrame(cv_rows).sort_values(['dataset_id', 'macro_f1_mean'], ascending=[True, False])
display(cv_results)

In [5]:
# STEP 5: Locked Holdout Test Evaluation
test_rows = []
for d_id, part in splits.items():
    ranked = cv_results[cv_results['dataset_id'] == d_id]
    best_name = ranked.iloc[0]['model']
    model = sklearn.base.clone(MODELS[best_name])
    model.fit(part['train']['text'], part['train']['label'])
    preds = model.predict(part['test']['text'])
    acc = accuracy_score(part['test']['label'], preds)
    macro_f1 = f1_score(part['test']['label'], preds, average='macro')
    test_rows.append({'dataset_id': d_id, 'best_model': best_name, 'accuracy': acc, 'macro_f1': macro_f1})

display(pd.DataFrame(test_rows))

In [6]:
# STEP 6: Selective Prediction & Review Routing
def classify_and_route(model, subject, body):
    text = f'subject: {subject.strip()}\nbody: {body.strip()}'
    predicted = model.predict([text])[0]
    low_margin = False
    mandatory_review = (predicted == 'urgent_action' or low_margin)
    return {
        'subject': subject, 'body': body, 'text': text,
        'predicted_class': predicted, 'mandatory_review': mandatory_review
    }

sample_route = classify_and_route(MODELS['multinomial_nb'].fit(splits['business_intent']['train']['text'], splits['business_intent']['train']['label']), 'Project Update', 'Please check status.')
print(sample_route)

In [7]:
# STEP 7: PII Redaction & Conditional Response Draft Generation
EMAIL_RE = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b')
PHONE_RE = re.compile(r'(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)')

def redact_pii(text):
    return PHONE_RE.sub('[PHONE_REDACTED]', EMAIL_RE.sub('[EMAIL_REDACTED]', text))

def generate_draft(prediction, sender='[Sender]', signature='[Your Name]'):
    p_class = prediction['predicted_class']
    if p_class == 'spam':
        return {'status': 'suppressed', 'reason': 'No draft for spam.', 'draft': None}
    safe_sub = redact_pii(prediction['subject'])
    draft = f'Subject: Re: {safe_sub}\n\nDear {sender},\n\nThank you for your message regarding: {p_class}. We have logged your request.\n\nBest regards,\n{signature}'
    return {'status': 'generated', 'draft': draft}

draft_res = generate_draft(sample_route)
print(draft_res['draft'])